In [ ]:
import numpy as np
import pandas as pd
import json
import os
from typing import Dict, List, Tuple, Any
from dataclasses import dataclass
from collections import defaultdict
from pathlib import Path
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("📊 Data sampling tool loaded")


In [ ]:
# ================================
# Data structure definitions
# ================================

@dataclass
class PaperInfo:
    """Basic paper information"""
    paper_id: str
    title: str
    abstract: str
    authors: List[Dict[str, str]]  # List of dicts containing id, name, org
    venue: str
    year: int
    keywords: List[str]
    n_citation: int
    doc_type: str
    # Add missing fields
    references: List[str] = None
    page_start: str = ""
    page_end: str = ""
    lang: str = "en"
    volume: str = ""
    issue: str = ""
    issn: str = ""
    isbn: str = ""
    doi: str = ""
    url: List[str] = None
    # Add per-year citation fields
    citations_year_1: int = 0
    citations_year_2: int = 0
    citations_year_3: int = 0
    citations_year_4: int = 0
    citations_year_5: int = 0

print("📋 Data structure definitions complete")


In [ ]:
# ================================
# Data sampling class
# ================================

class DataSampler:
    """Data sampler - responsible for sampling the training, validation, and test sets from the raw data"""

    def __init__(self, data_path: str):
        self.data_path = data_path
        self.all_papers = self._load_papers()
        logger.info(f"Data loading complete, {len(self.all_papers)} papers in total")

    def _load_papers(self, max_papers: int = None) -> List[PaperInfo]:
        """Load paper data"""
        papers = []
        print(f"Loading data file: {self.data_path}")

        with open(self.data_path, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(f):
                if max_papers and line_num >= max_papers:
                    break

                try:
                    data = json.loads(line.strip())
                    paper = PaperInfo(
                        paper_id=data['id'],
                        title=data['title'],
                        abstract=data.get('abstract', ''),
                        authors=data.get('authors', []),
                        venue=data.get('venue', ''),
                        year=data.get('year', 2010),
                        keywords=data.get('keywords', []),
                        n_citation=data.get('n_citation', 0),
                        doc_type=data.get('doc_type', 'Unknown'),
                        # Add missing fields
                        references=data.get('references', []),
                        page_start=data.get('page_start', ''),
                        page_end=data.get('page_end', ''),
                        lang=data.get('lang', 'en'),
                        volume=data.get('volume', ''),
                        issue=data.get('issue', ''),
                        issn=data.get('issn', ''),
                        isbn=data.get('isbn', ''),
                        doi=data.get('doi', ''),
                        url=data.get('url', []),
                        # Add per-year citation fields
                        citations_year_1=data.get('citations_year_1', 0),
                        citations_year_2=data.get('citations_year_2', 0),
                        citations_year_3=data.get('citations_year_3', 0),
                        citations_year_4=data.get('citations_year_4', 0),
                        citations_year_5=data.get('citations_year_5', 0)
                    )
                    papers.append(paper)

                    if line_num % 100000 == 0 and line_num > 0:
                        print(f"Loaded {line_num + 1} papers")

                except json.JSONDecodeError as e:
                    print(f"Error parsing line {line_num + 1}: {e}")
                except Exception as e:
                    print(f"Error processing line {line_num + 1}: {e}")

        print(f"Data loading complete, {len(papers)} papers in total")
        return papers

    def split_papers_by_year(self) -> Tuple[List[PaperInfo], List[PaperInfo], List[PaperInfo]]:
        """Split the paper data by year"""
        train_papers = []  # 2018 and earlier
        eval_papers = []   # 2019
        test_papers = []   # 2020 and later

        for paper in self.all_papers:
            if paper.year <= 2018:
                train_papers.append(paper)
            elif paper.year == 2019:
                eval_papers.append(paper)
            elif paper.year == 2020:
                test_papers.append(paper)

        logger.info(f"Data split complete:")
        logger.info(f"  Training set (≤2018): {len(train_papers)} papers")
        logger.info(f"  Validation set (2019): {len(eval_papers)} papers")
        logger.info(f"  Test set (2020): {len(test_papers)} papers")

        return train_papers, eval_papers, test_papers

    def sample_papers(self, papers: List[PaperInfo], n_samples: int, seed: int = 42) -> List[PaperInfo]:
        """Randomly sample papers"""
        if len(papers) <= n_samples:
            logger.info(f"Data size ({len(papers)}) is less than or equal to the target sample count ({n_samples}); returning all data")
            return papers

        np.random.seed(seed)
        indices = np.random.choice(len(papers), n_samples, replace=False)
        sampled_papers = [papers[i] for i in indices]

        logger.info(f"Sampled {len(sampled_papers)} papers from {len(papers)} papers")
        return sampled_papers

    def save_sampled_data(self, papers: List[PaperInfo], output_path: str):
        """Save the sampled data"""
        os.makedirs(os.path.dirname(output_path), exist_ok=True)

        with open(output_path, 'w', encoding='utf-8') as f:
            for paper in papers:
                paper_dict = {
                    'id': paper.paper_id,
                    'title': paper.title,
                    'abstract': paper.abstract,
                    'authors': paper.authors,
                    'venue': paper.venue,
                    'year': paper.year,
                    'keywords': paper.keywords,
                    'n_citation': paper.n_citation,
                    'doc_type': paper.doc_type,
                    # Add missing fields
                    'references': paper.references or [],
                    'page_start': paper.page_start,
                    'page_end': paper.page_end,
                    'lang': paper.lang,
                    'volume': paper.volume,
                    'issue': paper.issue,
                    'issn': paper.issn,
                    'isbn': paper.isbn,
                    'doi': paper.doi,
                    'url': paper.url or [],
                    # Add per-year citation fields
                    'citations_year_1': paper.citations_year_1,
                    'citations_year_2': paper.citations_year_2,
                    'citations_year_3': paper.citations_year_3,
                    'citations_year_4': paper.citations_year_4,
                    'citations_year_5': paper.citations_year_5
                }
                f.write(json.dumps(paper_dict, ensure_ascii=False) + '\n')

        logger.info(f"Sampled data saved to: {output_path} ({len(papers)} papers)")

print("🎯 Data sampler class definition complete")


In [ ]:
# ================================
# Sampling execution function
# ================================

def perform_data_sampling(data_path: str, output_dir: str = "maindata/test2",
                         sample_sizes: Dict[str, int] = None):
    """Run data sampling and save the results"""

    if sample_sizes is None:
        sample_sizes = {
            'train': 10000,  # Number of training set samples
            'eval': 1000,    # Number of validation set samples
            'test': 1000     # Number of test set samples
        }

    print("🚀 Starting the data sampling pipeline")
    print("=" * 60)
    print(f"Raw data path: {data_path}")
    print(f"Output directory: {output_dir}")
    print(f"Sampling configuration: {sample_sizes}")
    print("=" * 60)

    try:
        # Initialize the sampler
        sampler = DataSampler(data_path)

        # Split the data by year
        train_papers, eval_papers, test_papers = sampler.split_papers_by_year()

        # Run sampling
        datasets = {
            'train': sampler.sample_papers(train_papers, sample_sizes['train'], seed=62),
            'eval': sampler.sample_papers(eval_papers, sample_sizes['eval'], seed=63),
            'test': sampler.sample_papers(test_papers, sample_sizes['test'], seed=64)
        }

        # Save the sampling results
        print("\n💾 Saving the sampling results...")
        for dataset_name, papers in datasets.items():
            if papers:  # Only save non-empty datasets
                output_path = os.path.join(output_dir, f"{dataset_name}.jsonl")
                sampler.save_sampled_data(papers, output_path)
            else:
                print(f"⚠️ Skipping empty dataset: {dataset_name}")

        # Output statistics
        print("\n📊 Sampling completion statistics:")
        print("-" * 40)
        for dataset_name, papers in datasets.items():
            print(f"{dataset_name.upper():<8}: {len(papers):>6} papers")

        print(f"\n✅ Data sampling complete! The sampled data has been saved to: {output_dir}")

        return datasets

    except Exception as e:
        print(f"❌ Data sampling failed: {e}")
        logger.exception("Detailed error information:")
        raise

print("⚙️ Sampling execution function definition complete")


In [ ]:
# ================================
# Main execution code
# ================================

if __name__ == "__main__":
    # Configuration parameters
    data_path = "feature_extraction/ai_papers_complete_all_years_20251003_011733_converted.jsonl"
    output_dir = "maindata/Openalex/test1"

    # Custom sample sizes (adjust as needed)
    sample_sizes = {
        'train': 10000,   # Number of training set samples (reduced for testing)
        'eval': 1000,     # Number of validation set samples
        'test': 1000      # Number of test set samples
    }

    # Run data sampling
    datasets = perform_data_sampling(data_path, output_dir, sample_sizes)

    print("\n🎉 The entire data sampling pipeline is complete!")
    print("📁 Sampled data files:")
    print(f"  • Training set: {output_dir}/train.jsonl")
    print(f"  • Validation set: {output_dir}/eval.jsonl")
    print(f"  • Test set: {output_dir}/test.jsonl")
    print("\n💡 You can now use data_processing.ipynb to process this sampled data")
